# Random Forest IDPS Model - Iteration 2

## Objective
Drop identifier features that cause data leakage and retrain the model using only behavioral features.

## Changes from Iteration 1
- Remove non-behavioral columns: Timestamp, Flow ID, Src IP, Dst IP, Source Port, Dst Port, Unnamed: 0
- Keep only behavioral features (packet lengths, IATs, flow duration, header flags, packet rates, etc.)
- Add cross-validation for stability assessment
- Compare performance with Iteration 1


In [ ]:
# === Imports ===
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score
)
from sklearn.preprocessing import RobustScaler, label_binarize
import joblib
import yaml

# === Display & paths ===
sns.set_theme()
pd.set_option("display.max_columns", 120)

REPORTS_DIR = Path("../reports")
FIGS_DIR = REPORTS_DIR / "figs"
MODELS_DIR = Path("../models")
CONFIG_DIR = Path("../config")

for d in [REPORTS_DIR, FIGS_DIR, MODELS_DIR, CONFIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# === Training config ===
RANDOM_STATE = 42
TEST_SIZE = 0.30
TARGET_CANDIDATES = ["label","Label"," Label","class","Class","attack","Attack","target","Target"]

# Optional scaling (RF doesn't need scaling, but RobustScaler helps if features are very skewed)
APPLY_SCALING = False
ROBUST_SCALER_QR = (5.0, 95.0)  # quantile range for RobustScaler

# Cross-validation settings
CV_FOLDS = 5


In [ ]:
# Load cleaned dataset
DATA_PATH = "../data/cleaned_cicddos2019_sample.csv"
df = pd.read_csv(DATA_PATH)
print(f"Original dataset shape: {df.shape}")
print(f"Original columns: {len(df.columns)}")
df.head(3)


In [ ]:
# === STEP 1: Identify and Drop Identifier Features ===
# Define identifier columns that cause data leakage
IDENTIFIER_COLUMNS = [
    'Unnamed: 0',      # Index column
    'Flow ID',         # Unique flow identifier
    ' Source IP',      # Source IP address
    ' Source Port',    # Source port
    ' Destination IP', # Destination IP address
    ' Destination Port', # Destination port
    ' Timestamp',      # Timestamp information
]

print("Identifier columns to be removed:")
for col in IDENTIFIER_COLUMNS:
    if col in df.columns:
        print(f"  ✓ {col}")
    else:
        print(f"  ✗ {col} (not found)")

# Remove identifier columns
df_behavioral = df.drop(columns=[col for col in IDENTIFIER_COLUMNS if col in df.columns])

print(f"\nAfter removing identifier columns:")
print(f"Shape: {df_behavioral.shape}")
print(f"Remaining columns: {len(df_behavioral.columns)}")
print(f"Removed {df.shape[1] - df_behavioral.shape[1]} identifier columns")


In [ ]:
# === STEP 2: Identify Target Column and Prepare Data ===
target_col = " Label"  # The actual column name with leading space
print("Target column:", target_col)

# Check target distribution
print("\nTarget class distribution:")
print(df_behavioral[target_col].value_counts().sort_index())

# Prepare features and target
X = df_behavioral.drop(columns=[target_col])
y = df_behavioral[target_col].astype(int)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Number of classes: {len(y.unique())}")


In [ ]:
# === STEP 3: Train-Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# Check class distribution in train/test sets
print("\nTraining set class distribution:")
print(y_train.value_counts().sort_index())
print("\nTest set class distribution:")
print(y_test.value_counts().sort_index())


In [ ]:
# === STEP 4: Data Preprocessing ===
# Coerce to numeric, set bad values to NaN
X_train = X_train.apply(pd.to_numeric, errors="coerce")
X_test = X_test.apply(pd.to_numeric, errors="coerce")

# Replace ±inf with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Check for missing values
print("Missing values in training set:")
missing_train = X_train.isnull().sum()
print(f"Columns with missing values: {(missing_train > 0).sum()}")
if (missing_train > 0).sum() > 0:
    print(missing_train[missing_train > 0])

print("\nMissing values in test set:")
missing_test = X_test.isnull().sum()
print(f"Columns with missing values: {(missing_test > 0).sum()}")
if (missing_test > 0).sum() > 0:
    print(missing_test[missing_test > 0])

# Median imputation (robust to skew)
imp = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imp.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(imp.transform(X_test), columns=X.columns, index=X_test.index)

# Optional: Robust scaling (RF doesn't require it, but safe if you want)
if APPLY_SCALING:
    scaler = RobustScaler(quantile_range=ROBUST_SCALER_QR)
    num_cols = X_train.columns  # all are numeric now
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

print("\nPreprocessed shapes:", X_train.shape, X_test.shape)


In [ ]:
# === STEP 5: Train Random Forest Model ===
print("Training Random Forest model...")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",   # helpful for class imbalance
    random_state=RANDOM_STATE
)

rf.fit(X_train, y_train)
print("Model training completed!")

# Get predictions and probabilities
pred_test = rf.predict(X_test)
proba_test = rf.predict_proba(X_test)

print(f"Prediction shape: {pred_test.shape}")
print(f"Probability shape: {proba_test.shape}")


In [ ]:
# === STEP 6: Model Evaluation ===
# Calculate metrics
acc = accuracy_score(y_test, pred_test)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred_test, average="weighted", zero_division=0)

print("=== Iteration 2 Results ===")
print(f"Accuracy: {acc:.4f}")
print(f"Weighted Precision: {prec:.4f}")
print(f"Weighted Recall: {rec:.4f}")
print(f"Weighted F1-score: {f1:.4f}")

# Detailed per-class performance
print("\n=== Per-Class Performance ===")
per_class_metrics = precision_recall_fscore_support(y_test, pred_test, average=None, zero_division=0)
classes = np.unique(y_test)

for i, cls in enumerate(classes):
    print(f"Class {cls}: Precision={per_class_metrics[0][i]:.3f}, Recall={per_class_metrics[1][i]:.3f}, F1={per_class_metrics[2][i]:.3f}")

# Classification report
print("\n=== Detailed Classification Report ===")
print(classification_report(y_test, pred_test, zero_division=0))


In [ ]:
# === STEP 7: Cross-Validation for Stability ===
print("Running 5-fold cross-validation...")

# Prepare full dataset for CV
X_full = X_train.copy()
y_full = y_train.copy()

# Cross-validation scores
cv_scores_accuracy = cross_val_score(rf, X_full, y_full, cv=CV_FOLDS, scoring='accuracy')
cv_scores_f1_weighted = cross_val_score(rf, X_full, y_full, cv=CV_FOLDS, scoring='f1_weighted')
cv_scores_recall_weighted = cross_val_score(rf, X_full, y_full, cv=CV_FOLDS, scoring='recall_weighted')
cv_scores_precision_weighted = cross_val_score(rf, X_full, y_full, cv=CV_FOLDS, scoring='precision_weighted')

print("=== Cross-Validation Results (5-fold) ===")
print(f"Accuracy:  {cv_scores_accuracy.mean():.4f} ± {cv_scores_accuracy.std():.4f}")
print(f"Precision: {cv_scores_precision_weighted.mean():.4f} ± {cv_scores_precision_weighted.std():.4f}")
print(f"Recall:    {cv_scores_recall_weighted.mean():.4f} ± {cv_scores_recall_weighted.std():.4f}")
print(f"F1-score:  {cv_scores_f1_weighted.mean():.4f} ± {cv_scores_f1_weighted.std():.4f}")

# Individual fold results
print("\n=== Individual Fold Results ===")
cv_results = pd.DataFrame({
    'Fold': range(1, CV_FOLDS + 1),
    'Accuracy': cv_scores_accuracy,
    'Precision': cv_scores_precision_weighted,
    'Recall': cv_scores_recall_weighted,
    'F1-Score': cv_scores_f1_weighted
})
print(cv_results.round(4))


In [ ]:
# === STEP 8: Confusion Matrix ===
cm = confusion_matrix(y_test, pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cbar=True, cmap='Blues')
plt.title("Confusion Matrix - Iteration 2 (Behavioral Features Only)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(FIGS_DIR / "confusion_matrix_iteration2.png", dpi=120)
plt.show()


In [ ]:
# === STEP 9: Feature Importance Analysis ===
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top20 = importances.head(20)

plt.figure(figsize=(10, 8))
top20[::-1].plot(kind="barh")
plt.title("Top-20 Feature Importances - Iteration 2 (Behavioral Features Only)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIGS_DIR / "feature_importance_top20_iteration2.png", dpi=120)
plt.show()

print("Top 10 Most Important Features:")
print(top20.head(10))

# Save feature importance to CSV
top20.to_csv(REPORTS_DIR / "feature_importance_top20_iteration2.csv")


In [ ]:
# === STEP 10: Save Model and Results ===
# Save model
model_path = MODELS_DIR / "rf_multiclass_iteration2.pkl"
joblib.dump(rf, model_path)

# Save config for serving
config = {
    "model_path": str(model_path),
    "feature_columns": list(X.columns),
    "target": target_col,
    "classes": [int(cls) for cls in np.unique(y)],
    "removed_identifier_columns": [col for col in IDENTIFIER_COLUMNS if col in df.columns],
    "notes": "RandomForest (multiclass, iteration2), behavioral features only, class_weight=balanced, median-imputed, "
             + ("robust-scaled" if APPLY_SCALING else "no-scaling")
}
with open(CONFIG_DIR / "model_config_iteration2.yaml", "w") as f:
    yaml.safe_dump(config, f)

# Save comprehensive metrics
metrics = {
    "iteration": 2,
    "description": "Behavioral features only (identifier features removed)",
    "data_info": {
        "original_shape": list(df.shape),
        "final_shape": list(df_behavioral.shape),
        "removed_columns": len(df.columns) - len(df_behavioral.columns),
        "n_features_used": len(X.columns)
    },
    "performance": {
        "accuracy": float(acc),
        "weighted_precision": float(prec),
        "weighted_recall": float(rec),
        "weighted_f1": float(f1)
    },
    "cross_validation": {
        "cv_folds": CV_FOLDS,
        "accuracy_mean": float(cv_scores_accuracy.mean()),
        "accuracy_std": float(cv_scores_accuracy.std()),
        "f1_mean": float(cv_scores_f1_weighted.mean()),
        "f1_std": float(cv_scores_f1_weighted.std()),
        "recall_mean": float(cv_scores_recall_weighted.mean()),
        "recall_std": float(cv_scores_recall_weighted.std()),
        "precision_mean": float(cv_scores_precision_weighted.mean()),
        "precision_std": float(cv_scores_precision_weighted.std())
    }
}

with open(REPORTS_DIR / "metrics_iteration2.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("=== Model and Results Saved ===")
print(f"Model saved to: {model_path}")
print(f"Config saved to: {CONFIG_DIR / 'model_config_iteration2.yaml'}")
print(f"Metrics saved to: {REPORTS_DIR / 'metrics_iteration2.json'}")
print(f"Figures saved to: {FIGS_DIR}")


In [ ]:
# === STEP 11: Comparison with Iteration 1 ===
# Load Iteration 1 metrics if available
iteration1_metrics_path = REPORTS_DIR / "metrics.json"

if iteration1_metrics_path.exists():
    with open(iteration1_metrics_path, 'r') as f:
        iteration1_metrics = json.load(f)
    
    print("=== Performance Comparison: Iteration 1 vs Iteration 2 ===")
    print(f"{'Metric':<20} {'Iteration 1':<15} {'Iteration 2':<15} {'Difference':<15}")
    print("-" * 65)
    
    # Compare key metrics
    metrics_to_compare = ['accuracy', 'weighted_precision', 'weighted_recall', 'weighted_f1']
    
    for metric in metrics_to_compare:
        iter1_val = iteration1_metrics.get(metric, 0)
        iter2_val = metrics['performance'][metric]
        diff = iter2_val - iter1_val
        
        print(f"{metric:<20} {iter1_val:<15.4f} {iter2_val:<15.4f} {diff:+.4f}")
    
    print("\n=== Analysis ===")
    if metrics['performance']['accuracy'] < iteration1_metrics.get('accuracy', 1):
        print("✓ Expected: Accuracy decreased after removing identifier features")
        print("  This indicates the model was previously relying on data leakage")
    else:
        print("⚠ Unexpected: Accuracy increased or stayed the same")
        print("  This might indicate the identifier features were not causing leakage")
    
    print(f"\n✓ Model now uses only {metrics['data_info']['n_features_used']} behavioral features")
    print(f"✓ Removed {metrics['data_info']['removed_columns']} identifier columns")
    print(f"✓ Cross-validation shows stability: F1 = {cv_scores_f1_weighted.mean():.4f} ± {cv_scores_f1_weighted.std():.4f}")
    
else:
    print("Iteration 1 metrics not found. Run Iteration 1 first for comparison.")
    print("\n=== Current Iteration 2 Results ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print(f"Cross-validation F1: {cv_scores_f1_weighted.mean():.4f} ± {cv_scores_f1_weighted.std():.4f}")


In [ ]:
# === STEP 12: Sample Predictions ===
# Test model predictions on a sample
sample = X_test.iloc[:5].copy()
predictions = rf.predict(sample)
probabilities = rf.predict_proba(sample)

# Get the class names
classes = np.unique(y)
class_names = [f"Class_{cls}" for cls in classes]

# Create a results dataframe
results_df = pd.DataFrame({
    "Predicted_Class": predictions,
    "True_Class": y_test.iloc[:5].values
})

# Add probability scores for each class
for i, class_name in enumerate(class_names):
    results_df[f"Prob_{class_name}"] = probabilities[:, i]

print("=== Sample Predictions (Iteration 2) ===")
print(results_df)

# Show prediction accuracy for this sample
correct_predictions = (predictions == y_test.iloc[:5].values).sum()
print(f"\nSample accuracy: {correct_predictions}/5 = {correct_predictions/5:.2%}")
